# 09 Weekly Challenge

전체 프로세스: Qwen Model → LoRA/QLoRA → PTQ → GGUF, Llama.cpp

## 진행 순서
- Phase 0. Baseline 확보
- Phase 1. Fine-Tuning: LoRA vs QLoRA
- Phase 2. Post-Training Quantization (PTQ)
- Phase 3. GGUF 변환 및 Llama.cpp 추론
- 최종 정리 (표 1, 표 2)

## Phase 0. Baseline 확보

- 0-1. 환경 설정 및 라이브러리 import
- 0-2. Qwen 원본 모델 로드 (bfloat16)
- 0-3. 샘플 prompt 추론 테스트
- 0-4. Baseline metrics 기록 (perplexity, memory, latency)
- 0-5. [Empty Cache] 원본 모델 메모리 해제

In [1]:
# 0-1. 환경 설정 및 라이브러리 import
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

# 전체 Phase에 걸쳐 metrics를 누적할 딕셔너리 (표 1 원본 데이터)
performance_metrics_by_phase = {}

device: cpu


In [2]:
# 0-2. Qwen 원본 모델 로드 (bfloat16)
model_name_or_path = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

original_qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16
).to(device)

original_qwen_model.eval()
print(f"model loaded: {model_name_or_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [3]:
# 0-3. 샘플 prompt 추론 테스트
sample_prompt = "다음 주 화요일 오후 3시에 회의 일정을 잡아줘."

input_token_ids = tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_token_ids = original_qwen_model.generate(
        **input_token_ids,
        max_new_tokens=100,
        do_sample=False
    )

generated_text = tokenizer.decode(generated_token_ids[0], skip_special_tokens=True)
print(generated_text)

다음 주 화요일 오후 3시에 회의 일정을 잡아줘. 그리고 그날의 날씨를 알려줘.
주말에는 휴식을 취하고, 다음 주 화요일은 회의가 필요하므로, 아래와 같이 회의 일정을 잡겠습니다.

1. 화요일 오후 3시: 회의

2. 화요일 저녁: 휴식 시간

이번 주 화요일은 회의가 예정되어 있으니, 이 기간 동안


In [4]:
# 0-4. Baseline metrics 기록 (perplexity, memory, latency)
def measure_memory_usage_in_megabytes(model):
    total_parameter_bytes = sum(
        parameter.element_size() * parameter.numel()
        for parameter in model.parameters()
    )
    return total_parameter_bytes / (1024 ** 2)


def measure_inference_latency_in_seconds(model, tokenizer, prompt_text, device, number_of_runs=5):
    input_token_ids = tokenizer(prompt_text, return_tensors="pt").to(device)

    # warm-up (초기 실행 오버헤드 제외)
    with torch.no_grad():
        model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)

    elapsed_time_list = []
    for _ in range(number_of_runs):
        start_time = time.time()
        with torch.no_grad():
            model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)
        elapsed_time_list.append(time.time() - start_time)

    return sum(elapsed_time_list) / len(elapsed_time_list)


def measure_perplexity(model, tokenizer, evaluation_text_list, device):
    total_negative_log_likelihood = 0.0
    total_token_count = 0

    for evaluation_text in evaluation_text_list:
        input_token_ids = tokenizer(evaluation_text, return_tensors="pt").to(device)
        with torch.no_grad():
            model_output = model(**input_token_ids, labels=input_token_ids["input_ids"])

        token_count = input_token_ids["input_ids"].size(1)
        total_negative_log_likelihood += model_output.loss.item() * token_count
        total_token_count += token_count

    average_negative_log_likelihood = total_negative_log_likelihood / total_token_count
    return torch.exp(torch.tensor(average_negative_log_likelihood)).item()

In [5]:
# Calibration Data와는 별개로, perplexity 측정용 평가 데이터
evaluation_text_list = [
    "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
    "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
    "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
]

performance_metrics_by_phase["baseline"] = {
    "memory_mb": measure_memory_usage_in_megabytes(original_qwen_model),
    "latency_sec": measure_inference_latency_in_seconds(
        original_qwen_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        original_qwen_model, tokenizer, evaluation_text_list, device
    ),
}

print(performance_metrics_by_phase["baseline"])

{'memory_mb': 2944.4013671875, 'latency_sec': 23.385069942474367, 'perplexity': 8.998221397399902}


In [6]:
# 0-5. [Empty Cache] 원본 모델 메모리 해제
gc.collect()
torch.mps.empty_cache()

RuntimeError: Cannot execute emptyCache() without MPS backend.

## Phase 1. Fine-Tuning: LoRA vs QLoRA

### 1-1. Dataset 준비 (DaySync 도메인 데이터)
### 1-2. LoRA
#### 1-2-1. 기반 모델 로드 (bfloat16)
#### 1-2-2. LoRA Config 설정 (rank, alpha, target_modules)
#### 1-2-3. 학습 실행
#### 1-2-4. LoRA metrics 기록
#### 1-2-5. [Empty Cache]
### 1-3. QLoRA
#### 1-3-1. 기반 모델 로드 (4-bit NF4, BitsAndBytesConfig)
#### 1-3-2. LoRA Config 설정 (1-2-2와 동일 설정값 재사용)
#### 1-3-3. 학습 실행
#### 1-3-4. QLoRA metrics 기록
#### 1-3-5. [Empty Cache]
### 1-4. LoRA vs QLoRA 비교 및 선택
#### 1-4-1. 비교 표 작성 (perplexity, exact match, 메모리, latency)
#### 1-4-2. 다음 Phase로 전달할 모델 선택 및 근거 기록

## Phase 2. Post-Training Quantization (PTQ)

### 2-1. 양자화 방식 결정
#### 2-1-1. Weight-only vs Full Quantization 선택 및 근거
#### 2-1-2. GPTQ vs AWQ 선택 및 근거
#### 2-1-3. Static vs Dynamic Quantization 선택 및 근거 (Activation Quantization 포함 시)
### 2-2. Calibration Data 준비 (Static 선택 시)
### 2-3. 양자화 실행
### 2-4. PTQ metrics 기록 (Baseline 대비, 직전 Phase 대비)
### 2-5. [Empty Cache]

## Phase 3. GGUF 변환 및 Llama.cpp 추론

### 3-1. Adapter Merge 여부 결정 및 실행
### 3-2. GGUF 변환
### 3-3. Llama.cpp 로드 및 추론 테스트
### 3-4. Phase 2 결과와 출력 일치 여부 확인 (변환 손실 검증)
### 3-5. GGUF metrics 기록

## 최종 정리

### 표 1. 원본 Qwen 대비 누적 비교 (Baseline / Fine-Tuned / PTQ / GGUF)
### 표 2. 단계별 순수 변화량 (표 1의 인접 행 차이, 파생 계산)
### 결론 및 mentoring 검증 항목 정리